# Notebook 3: Key instance identification

In this notebook, we consider a problem specific to multi-instance learning (MIL): key instance detection (KID). The goal of KID is to identify the instances that are most relevant to the predicted property. For example, in a multi-conformer bioactivity model, KID aims to identify the conformer most strongly associated with the predicted biological activity, often referred to as the bioactive conformer.

In this notebook, we explore how to access instance-weight prediction functionality and how to evaluate KID performance.

In [ ]:
import pickle
import random

import numpy as np
import pandas as pd

### 1. Data load

As a dataset for this notebook, we use a manually constructed benchmark in which bioactive conformers are defined as those satisfying predefined pharmacophore patterns (bioactivity triggers). Each bag contains up to 20 conformers per molecule. Active conformers may match different numbers of manually designed pharmacophore templates; consequently, the greater the number of matched pharmacophores, the higher the assigned activity of the conformer. All remaining conformers are considered inactive and do not match any pharmacophore template.

The task is formulated as a regression problem at the bag level. The target value for each bag is defined as the maximum number of matched pharmacophores across all conformers in the bag, ranging from 1 to 7.

In [ ]:
import importlib.util

from huggingface_hub import hf_hub_download
from qsarmil.conformer import split_into_conformers


def load_module_from_hf(repo_id, filename, module_name, repo_type="dataset"):
    """Download a .py file from an HF repo and import it as a module, without touching sys.path."""
    path = hf_hub_download(repo_id, filename=filename, repo_type=repo_type)
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

In [ ]:
REPO_ID = "KagakuLab/QSARmil"

pkl_path = hf_hub_download(REPO_ID, filename="notebooks/train_conf.pkl", repo_type="dataset")
with open(pkl_path, "rb") as f:
    data_train = pickle.load(f)

pkl_path = hf_hub_download(REPO_ID, filename="notebooks/test_conf.pkl", repo_type="dataset")
with open(pkl_path, "rb") as f:
    data_test = pickle.load(f)

In [ ]:
# molecules
mols_train = [i[1] for i in data_train]
mols_test = [i[1] for i in data_test]

# conformers - split each already-embedded Mol into a bag of
# single-conformer Mol objects (a plain list, no wrapper type)
confs_train = [split_into_conformers(i) for i in mols_train]
confs_test = [split_into_conformers(i) for i in mols_test]

# property
y_train = [i[3] for i in data_train]
y_test = [i[3] for i in data_test]

# active conformers
idx_train = [i[2] for i in data_train]
idx_test = [i[2] for i in data_test]

### 2. Descriptor calculation

Many different types of descriptors can be used to encode conformers; however, the choice of descriptors should satisfy a few key requirements. The descriptors must be able to distinguish between different conformers, meaning that their vector representations should vary sufficiently across conformers. Also, the descriptors should be relevant to the target property, ensuring that they capture chemically meaningful information for the prediction task.

In [ ]:
# 3D descriptors
from qsarmil.descriptor.rdkit import RDKitGEOM, RDKitAUTOCORR, RDKitRDF, RDKitMORSE, RDKitWHIM, RDKitGETAWAY
from molfeat.calc import Pharmacophore3D, USRDescriptors, ElectroShapeDescriptors
from qsarmil.descriptor.wrapper import DescriptorWrapper
from milearn.preprocessing import BagMinMaxScaler

In [ ]:
desc_calc = DescriptorWrapper(Pharmacophore3D(factory='pmapper'), verbose=True)

In [ ]:
# 1. Calculate descriptors
x_train = desc_calc.run(confs_train)
x_test = desc_calc.run(confs_test)

In [ ]:
# 2. Scale descriptors
scaler = BagMinMaxScaler()
scaler.fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

### 3. Model building

Not all MIL methods are capable of identifying key instances, and different approaches rely on different mechanisms for KID. One of the most widely used strategies is the attention-based mechanism, where multi-instance neural networks are extended with an attention module that assigns weights to individual instances.

In **QSARmil**, several methods implement this attention-based idea. In addition to standard prediction outputs ``model.predict(x)``, these methods also provide a instance weight prediction ``model.get_instance_weights(x)`` method, which returns a list of weights corresponding to each instance in the input bag.

In [ ]:
# Network hparams
from milearn.network.module.hopt import DEFAULT_PARAM_GRID

# MIL networks (with instance weight prediction)
from milearn.network.regressor import (AdditiveAttentionNetworkRegressor, 
                                       SelfAttentionNetworkRegressor,
                                       HopfieldAttentionNetworkRegressor,
                                       DynamicPoolingNetworkRegressor)

In [ ]:
model = DynamicPoolingNetworkRegressor()
model.hopt(x_train_scaled, y_train, param_grid=DEFAULT_PARAM_GRID, verbose=True)
model.fit(x_train_scaled, y_train)

### 4. KID accuracy evaluation

To evaluate **key instance detection (KID)** performance, we use a function that compares predicted instance importance scores with ground-truth key instance annotations.

The inputs are structured as follows: ``y_true`` is a binary indicator vector where each element corresponds to an instance in the bag and active bits are the true key instances. ``y_pred`` is a vector of the same shape, where each element represents the predicted weight assigned to the corresponding instance.

The evaluation returns two metrics. The first is the empirical KID accuracy, which measures how often the top-ranked predicted instances correctly identify at least one true key instance. The second is the expected KID accuracy, which serves as a random-selection baseline and reflects the probability of hitting a key instance by chance given the number of positives and the bag size.

In [ ]:
# KID accuracy metrics
# kid_accuracy is a small, notebook-only utility (not part of the qsarmil package),
# so it's pulled in from the same HF dataset repo as the benchmark data above.
from sklearn.metrics import r2_score

metrics = load_module_from_hf(REPO_ID, filename="notebooks/metrics.py", module_name="metrics")
kid_accuracy = metrics.kid_accuracy

In [ ]:
# convert indeces to binary vector
def idx_to_binary(x, k):
    y = []
    for bag, ki in zip(x, k):
        n = len(bag)
        label = np.zeros(n, dtype=int)
        label[ki] = 1
        y.append(label)
    return y

In [ ]:
# key binary 
keys_train = idx_to_binary(confs_train, idx_train)
keys_test = idx_to_binary(confs_test, idx_test)

In [ ]:
y_pred = model.predict(x_test_scaled)
w_pred = model.get_instance_weights(x_test_scaled)
w_pred = [w.flatten() for w in w_pred]

In [ ]:
top_n = 1

print(f"All molecules: {len(y_test)}")
print(f"Prediction accuracy: {r2_score(y_test, y_pred):.2f}")

acc, exp = kid_accuracy(keys_test, w_pred, top_n=top_n)
print(f"KID prediction accuracy: {acc:.2f}")
print(f"KID baseline accuracy: {exp:.2f}")

idx_7 = []
for n, y in enumerate(y_test):
    if y == 7:
        idx_7.append(n)
keys_test_7 = [keys_test[i] for i in idx_7]
w_pred_7 = [w_pred[i] for i in idx_7]

print(f"\nActive molecules: {len(idx_7)}")

acc, exp = kid_accuracy(keys_test_7, w_pred_7, top_n=top_n)
print(f"KID prediction accuracy: {acc:.2f}")
print(f"KID baseline accuracy: {exp:.2f}")

### 5. KID prediction visualization

The conformers and their corresponding predicted instance weights can be visually inspected using the utility function provided below. This allows manual exploration of how the model assigns importance scores across different conformations, providing an intuitive way to interpret key instance detection results.

This step needs **py3Dmol** for the 3D rendering (`pip install py3Dmol`) - it's not a dependency of ``qsarmil`` itself, only of this visualization helper.


In [ ]:
visualization = load_module_from_hf(REPO_ID, filename="notebooks/visualization.py", module_name="visualization")
visualize_conformers_grid = visualization.visualize_conformers_grid

In [ ]:
# most active molecules indeces
print(idx_7)

In [ ]:
N = 45 # choose molecule index from the list above

print(y_pred[N])
print(y_test[N])

In [ ]:
visualize_conformers_grid(mols_test[N], w_pred[N], idx_test[N], top_n=5, sort_by_weight=True)